**Cell 1: Environment Setup & Data Extraction**

In [ ]:
import os
from google.colab import drive
import zipfile

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Image Segmentation Project/Project/Task 7/Anomaly_Validation_Datasets.zip'
EXTRACT_DIR = '/content/datasets'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    print("Unzipping datasets... this might take a minute.")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Unzip complete!")
else:
    print("Datasets already extracted.")

Mounted at /content/drive
Unzipping datasets... this might take a minute.
Unzip complete!


**Cell 2: The Reusable Metric Engine**

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import auc, roc_curve, precision_recall_curve

def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    """
    Calculates AuPRC and FPR95 for anomaly segmentation.
    y_true: Ground truth numpy array (0 = Normal, 1 = Anomaly, 255 = Ignore)
    y_scores: Anomaly score numpy array (Higher = more anomalous)
    """
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()

    # Filter out the 'ignore' regions (standard approach)
    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]

    # Safety check
    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0

    # 1. AuPRC
    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)

    # 2. FPR95
    fpr, tpr, thresholds = roc_curve(y_true_filtered, y_scores_filtered)

    idx = np.where(tpr >= 0.95)[0][0]
    fpr95 = fpr[idx]

    return auprc, fpr95

**Cell 3: The ERFNet Anomaly Score Extractors**

In [ ]:
def get_msp_score(logits):
    """ MSP Score: 1 - max(softmax(logits)) """
    probs = F.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    anomaly_score = 1.0 - max_probs
    return anomaly_score.cpu().numpy()

def get_maxlogit_score(logits):
    """ MaxLogit Score: -max(logits) """
    max_logits, _ = torch.max(logits, dim=1)
    anomaly_score = -max_logits
    return anomaly_score.cpu().numpy()

def get_max_entropy_score(logits):
    """ Max Entropy Score: -sum(p * log(p)) """
    probs = F.softmax(logits, dim=1)
    # Add a tiny epsilon to prevent log(0) resulting in NaN
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
    return entropy.cpu().numpy()

**Cell 4: The Main Evaluation Loop & Formatted Output**

In [ ]:
import sys, os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import auc, roc_curve, precision_recall_curve

SHARED_FOLDER_NAME = 'Image Segmentation Project/Project/Task 7'
EVAL_FOLDER_PATH = '<BASELINE_REPO_PATH>/eval'
sys.path.append(EVAL_FOLDER_PATH)

from erfnet import ERFNet
from transform import ToLabel

def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()

    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]

    # Force binarization: 0 stays 0, anything else (like 2) becomes 1
    y_true_filtered = (y_true_filtered > 0).astype(int)

    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0

    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]

    return auprc, fpr[idx]

def get_msp_score(logits):
    probs = F.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    return (1.0 - max_probs).cpu().numpy()

def get_maxlogit_score(logits):
    max_logits, _ = torch.max(logits, dim=1)
    return (-max_logits).cpu().numpy()

def get_max_entropy_score(logits):
    probs = F.softmax(logits, dim=1)

    # Calculate raw entropy (with 1e-8 for numerical stability to prevent log(0) crashes)
    raw_entropy = torch.sum(-probs * torch.log(probs + 1e-8), dim=1)

    # Divide by log(C)
    num_classes = probs.shape[1]
    normalized_entropy = torch.div(raw_entropy, torch.log(torch.tensor(num_classes, dtype=torch.float32, device=logits.device)))

    return normalized_entropy.cpu().numpy()

class CustomAnomalyDataset(Dataset):
    def __init__(self, root, input_transform=None, target_transform=None):
        self.images_dir = os.path.join(root, 'images')
        self.masks_dir = os.path.join(root, 'labels_masks')
        image_files = sorted([f for f in os.listdir(self.images_dir) if not f.startswith('.')])
        self.samples = []
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_files = [m for m in os.listdir(self.masks_dir) if m.startswith(stem + '.')]
            if mask_files:
                self.samples.append((img_file, mask_files[0]))
        self.input_transform = input_transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        img_name, mask_name = self.samples[index]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        with open(img_path, 'rb') as f: image = Image.open(f).convert('RGB')
        with open(mask_path, 'rb') as f: mask = Image.open(f).convert('P')
        if self.input_transform: image = self.input_transform(image)
        if self.target_transform: mask = self.target_transform(mask)
        return image, mask, img_name, mask_name

    def __len__(self):
        return len(self.samples)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 1

print("Loading ERFNet...")
model = ERFNet(20).to(device)

weights_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/erfnet_pretrained.pth'
checkpoint = torch.load(weights_path, map_location=device)

cleaned_state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
model.load_state_dict(cleaned_state_dict, strict=False)
model.eval()
print("Weights loaded successfully!")

input_transform_fn = transforms.Compose([transforms.ToTensor()])
target_transform_fn = ToLabel()

Loading ERFNet...
Weights loaded successfully!


In [ ]:
def evaluate_single_dataset(dataset_name, base_dir='/content/datasets/Validation_Dataset'):
    print(f"\n--- Evaluating on {dataset_name} ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    all_gt, all_msp, all_maxlogit, all_entropy = [], [], [], []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        logits = model(images)
        masks = masks.squeeze(1).numpy()

        # Memory optimization downcasting
        all_gt.append(masks.astype(np.uint8))
        all_msp.append(get_msp_score(logits).astype(np.float16))
        all_maxlogit.append(get_maxlogit_score(logits).astype(np.float16))
        all_entropy.append(get_max_entropy_score(logits).astype(np.float16))

        del images, logits
        torch.cuda.empty_cache()

    print(f"\nCalculating metrics for {dataset_name}...")

    # Calculate and clear memory
    gt_concat = np.concatenate(all_gt)
    del all_gt

    msp_concat = np.concatenate(all_msp)
    del all_msp
    msp_auprc, msp_fpr95 = calculate_anomaly_metrics(gt_concat, msp_concat)
    del msp_concat

    maxlogit_concat = np.concatenate(all_maxlogit)
    del all_maxlogit
    ml_auprc, ml_fpr95 = calculate_anomaly_metrics(gt_concat, maxlogit_concat)
    del maxlogit_concat

    entropy_concat = np.concatenate(all_entropy)
    del all_entropy
    ent_auprc, ent_fpr95 = calculate_anomaly_metrics(gt_concat, entropy_concat)
    del entropy_concat
    del gt_concat

    return {
        "MSP": {"AuPRC": msp_auprc, "FPR95": msp_fpr95},
        "MaxLogit": {"AuPRC": ml_auprc, "FPR95": ml_fpr95},
        "Max Entropy": {"AuPRC": ent_auprc, "FPR95": ent_fpr95}
    }

In [ ]:
import pandas as pd

if 'results_table' not in locals():
    results_table = {}

datasets_to_run = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

with torch.no_grad():
    for d_name in datasets_to_run:
        results_table[d_name] = evaluate_single_dataset(d_name)

df_results = pd.DataFrame(results_table).round(4)

print("\nRisultati Finali (AuPRC / FPR95):")
print(df_results.to_string(float_format='{:.3f}'.format))

# Salvataggio su Drive in CSV
#csv_path = f'/content/drive/MyDrive/{SHARED_FOLDER_NAME}/Task7_Results.csv'
#df_results.to_csv(csv_path)
#print(f"\nTabella salvata in: {csv_path}")


--- Evaluating on RoadAnomaly21 ---


Processing: 100%|██████████| 10/10 [00:01<00:00,  7.13it/s]



Calculating metrics for RoadAnomaly21...

--- Evaluating on RoadObsticle21 ---


Processing: 100%|██████████| 30/30 [00:09<00:00,  3.12it/s]



Calculating metrics for RoadObsticle21...

--- Evaluating on FS_LostFound_full ---


...


Calculating metrics for FS_LostFound_full...

--- Evaluating on fs_static ---


Processing: 100%|██████████| 30/30 [00:07<00:00,  3.87it/s]



Calculating metrics for fs_static...

--- Evaluating on RoadAnomaly ---


Processing: 100%|██████████| 60/60 [00:06<00:00,  8.82it/s]



Calculating metrics for RoadAnomaly...

Risultati Finali (AuPRC / FPR95):
                                                          RoadAnomaly21                                                RoadObsticle21                                             FS_LostFound_full                                                    fs_static                                                 RoadAnomaly
MSP          {'AuPRC': 0.2396648881730059, 'FPR95': 0.7230623262246256}  {'AuPRC': 0.006669279249514333, 'FPR95': 0.8971199356019548}  {'AuPRC': 0.012968743569417837, 'FPR95': 0.7988975388025961}  {'AuPRC': 0.03485301084107169, 'FPR95': 0.6332891820600332}  {'AuPRC': 0.10773456324521115, 'FPR95': 0.884779165773229}
MaxLogit     {'AuPRC': 0.3189672793744036, 'FPR95': 0.7243994811316398}      {'AuPRC': 0.01178000489682611, 'FPR95': 0.8030758126825}   {'AuPRC': 0.025541032659355003, 'FPR95': 0.721658216951484}  {'AuPRC': 0.04235570985891827, 'FPR95': 0.6588934427491283}  {'AuPRC': 0.1330869883442768, 'FP